In [1]:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

def get_embedding(text, model="qwen3-embedding-0.6b-dwq"):
   text = text.replace("\n", " ")
   return client.embeddings.create(input=[text], model=model).data[0].embedding


## Section 1: Parse Schedule & Make Embeddings

In [2]:
import json
import re

with open("data/nicar-2026-schedule.json") as f:
    data = json.load(f)

# Build speaker id -> "First Last, Affiliation" lookup
speaker_lookup = {
    s["id"]: f"{s['first_name']} {s['last_name']}, {s['affiliation']}".strip(", ")
    for s in data["speakers"]
}

def strip_html(text):
    return re.sub(r"<[^>]+>", "", text or "").strip()

sessions = [
    {
        "title": s["session_title"],
        "time": f"{s['start_time'][11:16]} - {s['end_time'][11:16]}",
        "room": s.get("room", ""),
        "track": ", ".join(s.get("tracks", [])),
        "type": s.get("session_type", ""),
        "speakers": ", ".join(speaker_lookup.get(sid, sid) for sid in s.get("speakers", [])),
        "description": strip_html(s.get("description", "")),
    }
    for s in data["sessions"]
    if not s.get("canceled")
]

print(f"Loaded {len(sessions)} sessions")
print(f"\nExample session:")
for k, v in sessions[0].items():
    print(f"  {k}: {str(v)[:80]}")


Loaded 231 sessions

Example session:
  title: How to get data out of complex documents using AI
  time: 14:00 - 15:00
  room: White River D
  track: AI, Tools & Tech
  type: Demo
  speakers: Disha Raychaudhuri, Reuters, Duy Nguyen, The New York Times, Teresa Mondría Tero
  description: Learn how newsrooms get important information out of large volumes of documents 


In [3]:
embeddings = []
for i, session in enumerate(sessions):
    # Embed title + description for richer semantics
    text = session["title"] + ". " + session.get("description", "")
    print(f"[{i+1}/{len(sessions)}] Embedding: '{session['title'][:60]}'")
    embeddings.append(get_embedding(text))

print(f"\nDone! Each embedding has {len(embeddings[0])} dimensions.")


[1/231] Embedding: 'How to get data out of complex documents using AI'
[2/231] Embedding: 'Everyone can be (and should be) a housing reporter'
[3/231] Embedding: 'AI concepts you should know'
[4/231] Embedding: 'From delving into data to door knocking: Finding people behi'
[5/231] Embedding: 'Collaborating in an intergenerational workspace: let's get t'
[6/231] Embedding: 'Wanna get hired? Tales from the trenches'
[7/231] Embedding: 'Immigration data and how to make sense of it'
[8/231] Embedding: 'Utilizing new police misconduct and employment databases'
[9/231] Embedding: 'Using data to find sources'
[10/231] Embedding: 'Demo: The Gun Violence Data Hub'
[11/231] Embedding: 'Connect with skeptical and disengaged audiences ... with dat'
[12/231] Embedding: 'How to scrape and analyze sports data in R to tell stories t'
[13/231] Embedding: 'Customizing ggplot for yourself or your organization'
[14/231] Embedding: 'Fun with hypotheticals: Investigating your dubious data dile'
[15/231] Emb

## Section 2: Reduce to 2D with TSNE

In [5]:
from sklearn.manifold import TSNE
import numpy as np

embedding_matrix = np.array(embeddings)

# Reduce to 2 dimensions using t-SNE with tighter clusters
tsne = TSNE(n_components=2, random_state=42, perplexity=15, max_iter=1000)
coords_2d = tsne.fit_transform(embedding_matrix)

print("Original shape:", embedding_matrix.shape)
print("Reduced shape: ", coords_2d.shape)


Original shape: (231, 768)
Reduced shape:  (231, 2)


## Section 3: Cluster with HDBSCAN

In [6]:
from sklearn.cluster import HDBSCAN

db = HDBSCAN(min_cluster_size=3, min_samples=2)
cluster_labels = db.fit_predict(coords_2d)

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise = (cluster_labels == -1).sum()
print(f"Clusters found: {n_clusters}")
print(f"Noise points:   {n_noise}")
print(f"Cluster sizes:  {sorted([(l, (cluster_labels==l).sum()) for l in set(cluster_labels) if l != -1], key=lambda x: -x[1])}")


Clusters found: 36
Noise points:   17
Cluster sizes:  [(np.int64(22), np.int64(18)), (np.int64(1), np.int64(16)), (np.int64(8), np.int64(16)), (np.int64(32), np.int64(15)), (np.int64(24), np.int64(10)), (np.int64(2), np.int64(9)), (np.int64(6), np.int64(8)), (np.int64(29), np.int64(8)), (np.int64(15), np.int64(7)), (np.int64(35), np.int64(7)), (np.int64(4), np.int64(6)), (np.int64(17), np.int64(6)), (np.int64(19), np.int64(6)), (np.int64(21), np.int64(5)), (np.int64(23), np.int64(5)), (np.int64(25), np.int64(5)), (np.int64(34), np.int64(5)), (np.int64(5), np.int64(4)), (np.int64(10), np.int64(4)), (np.int64(14), np.int64(4)), (np.int64(26), np.int64(4)), (np.int64(28), np.int64(4)), (np.int64(0), np.int64(3)), (np.int64(3), np.int64(3)), (np.int64(7), np.int64(3)), (np.int64(9), np.int64(3)), (np.int64(11), np.int64(3)), (np.int64(12), np.int64(3)), (np.int64(13), np.int64(3)), (np.int64(16), np.int64(3)), (np.int64(18), np.int64(3)), (np.int64(20), np.int64(3)), (np.int64(27), np.int6

/Users/mehtad/Development/localmodels-nicar26/.venv/lib/python3.12/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


## Section 4: Name Clusters  with Local LLMs

In [ ]:
import json

# Group session titles by cluster (skip noise)
from collections import defaultdict
cluster_titles = defaultdict(list)
for i, label in enumerate(cluster_labels):
    if label != -1:
        cluster_titles[int(label)].append(sessions[i]["title"])

# Build a single prompt with all clusters
cluster_block = "\n\n".join(
    f"Cluster {cid}:\n" + "\n".join(f"  - {t}" for t in titles)
    for cid, titles in sorted(cluster_titles.items())
)

response = client.chat.completions.create(
    model="meta-llama-3.1-8b-instruct",
    messages=[
        {
            "role": "system",
            "content": (
                "You are a cartographer naming regions on a fantasy map of a data journalism conference. "
                "For each cluster, invent a whimsical geographic place name that captures what makes it "
                "DISTINCT from the other clusters — not just what the sessions are about, but what sets "
                "this group apart. Names should be 2-4 words using geographic suffixes like: "
                "Bay, Cape, Cove, Delta, Flats, Forest, Gorge, Gulf, Harbor, Heights, Hills, Isle, "
                "Junction, Lagoon, Land, Mesa, Moors, Mountains, Peninsula, Plateau, Prairie, Range, "
                "Rapids, Reach, Ridge, Shore, Sound, Strait, Summit, Vale, Valley, Wastes, Wilds. "
                "Every name must be unique. "
                "Respond with ONLY valid JSON: {\"0\": \"Name Here\", \"1\": \"Name Here\", ...}"
            ),
        },
        {
            "role": "user",
            "content": f"Name each of these clusters with a unique map region name:\n\n{cluster_block}",
        },
    ],
    temperature=0.7,
    max_tokens=2000,
)

raw = response.choices[0].message.content.strip()

# Strip markdown fences if present
if raw.startswith("```"):
    raw = "\n".join(raw.split("\n")[1:])
    raw = raw.rsplit("```", 1)[0].strip()

try:
    cluster_names = {int(k): v for k, v in json.loads(raw).items()}
except json.JSONDecodeError as e:
    print(f"JSON parse error: {e}")
    print(f"Raw response ({len(raw)} chars):\n{raw}")
    raise

for cluster_id, name in sorted(cluster_names.items()):
    print(f"Cluster {cluster_id:2d} ({len(cluster_titles[cluster_id]):2d} sessions): {name}")

## Section 5: Visualize on an Interactive Scatter Plot

In [8]:
import plotly.express as px
import pandas as pd
import numpy as np
import textwrap

def wrap_text(text, width=80):
    return "<br>".join(textwrap.wrap(text, width)) if text else ""

def cluster_label(l):
    if l == -1:
        return "Noise"
    return cluster_names.get(int(l), f"Cluster {l}")

df = pd.DataFrame({
    "x": coords_2d[:, 0],
    "y": coords_2d[:, 1],
    "title": [s["title"] for s in sessions],
    "track": [s.get("track", "Other") for s in sessions],
    "type": [s.get("type", "") for s in sessions],
    "time": [s.get("time", "") for s in sessions],
    "room": [s.get("room", "") for s in sessions],
    "speakers": [s.get("speakers", "") for s in sessions],
    "description": [wrap_text(s.get("description", "")) for s in sessions],
    "cluster": [cluster_label(l) for l in cluster_labels],
})

fig = px.scatter(
    df,
    x="x",
    y="y",
    color="cluster",
    hover_name="title",
    custom_data=["time", "room", "type", "speakers", "description", "cluster"],
    title="NICAR 2026 Semantic Map (DBSCAN Clusters, LLM-named)",
    width=1200,
    height=800,
    color_discrete_map={"Noise": "#cccccc"},
)

fig.update_traces(
    marker=dict(size=8, opacity=0.8),
    hovertemplate=(
        "<b>%{hovertext}</b><br>"
        "%{customdata[0]} · %{customdata[1]} · %{customdata[2]}<br>"
        "%{customdata[3]}"
        "<br><br>"
        "%{customdata[4]}"
        "<br><br>"
        "<i>#%{customdata[5]}</i>"
        "<extra></extra>"
    ),
)

# Add centroid annotations for each named cluster
for cluster_id, name in cluster_names.items():
    mask = cluster_labels == cluster_id
    cx = coords_2d[mask, 0].mean()
    cy = coords_2d[mask, 1].mean()
    fig.add_annotation(
        x=cx,
        y=cy,
        text=f"<b>{name}</b>",
        showarrow=False,
        font=dict(size=11, color="black"),
        bgcolor="rgba(0,0,0,0)",
        borderpad=3,
    )

fig.update_layout(
    xaxis_title="t-SNE Dimension 1",
    yaxis_title="t-SNE Dimension 2",
    showlegend=False,
    hoverlabel=dict(bgcolor="white", font_size=12, namelength=-1),
)

fig.show()
